
Alaska FlexiMORP Complete Analysis - Test Code


SETUP AND IMPORTS


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import json
from datetime import datetime
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set up 
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
# Clear module cache
modules_to_remove = [k for k in sys.modules.keys() if k.startswith('fleximorpv2')]
for module in modules_to_remove:
    del sys.modules[module]

# FlexiMORP imports
from fleximorpv2.config import load_config
from fleximorpv2.baseline_optimization import BaselineOptimization
from fleximorpv2.uncertainty_analysis import UncertaintyAnalysis  
from fleximorpv2.flexible_design import FlexibleDesign
from fleximorpv2.sensitivity_analysis import SensitivityAnalysis
from fleximorpv2.graphics import GraphicsEngine
from fleximorpv2.utils.data_loader import APIDataLoader

print("✅ All imports successful - FlexiMORP v2 Alaska Analysis Ready!")


In [ ]:
# Load Alaska site configuration
try:
    config = load_config("alaska")
    print(f"✅ Configuration loaded successfully")
    print(f"   Site: {config.name}")
    print(f"   Coordinates: {config.coordinates}")
    print(f"   Technologies: {config.get_enabled_technologies()}")
except Exception as e:
    print(f"❌ Error loading config: {e}")

In [ ]:
print(f"\n{'='*60}")
print("STEP 1: BASELINE OPTIMIZATION")
print('='*60)

try:
    baseline_opt = BaselineOptimization(config)

    print("Running baseline optimization...")
    baseline_results = baseline_opt.optimize(
        target_type='production',
        target_value=200000,  # kWh target
        method='differential_evolution'
    )

    # optimal_design contains the design variables needed by downstream analysis steps
    optimal_design = baseline_results.optimal_design

    # base_design is a summary dict used for display and the executive summary
    base_design = {
        'lcoe': baseline_results.financial_metrics['lcoe'],
        'npv': baseline_results.financial_metrics['npv'],
        'total_capacity': baseline_results.technical_metrics['total_capacity'],
        'annual_energy': baseline_results.technical_metrics['annual_energy'],
        'wind_capacity': baseline_results.technology_capacities.get('wind', 0),
        'solar_capacity': baseline_results.technology_capacities.get('solar', 0),
        'hydro_capacity': baseline_results.technology_capacities.get('hydro', 0),
        'capacity_factor': baseline_results.technical_metrics['capacity_factor']
    }

    print("Baseline optimization completed")
    print(f"   Optimal LCOE: {base_design['lcoe']:.1f}/MWh")
    print(f"   Total Capacity: {base_design['total_capacity']:.2f} MW")
    print(f"   NPV: {base_design['npv']/1e6:.1f}M")
    print(f"   Annual Energy: {base_design['annual_energy']:.0f} MWh")
    print(f"   Wind: {base_design['wind_capacity']:.2f} MW  Solar: {base_design['solar_capacity']:.2f} MW  Hydro: {base_design['hydro_capacity']:.2f} MW")

except Exception as e:
    print(f"Baseline optimization failed: {e}")
    import traceback; traceback.print_exc()

    # Fallback summary dict for display
    base_design = {
        'lcoe': 95.0, 'npv': 25e6, 'total_capacity': 1.2,
        'annual_energy': 4200, 'wind_capacity': 0.6,
        'solar_capacity': 0.4, 'hydro_capacity': 0.1,
        'capacity_factor': 0.40
    }
    # Fallback optimal_design dict with the variable keys analysis modules expect
    optimal_design = {
        'wind_capacity': 0.6, 'solar_capacity': 0.4, 'hydro_capacity': 0.1,
        'platform_area': 6000, 'water_depth': 15, 'distance_to_shore': 0.5
    }
    print("Using fallback values for testing")

In [ ]:
print(f"\n{'='*60}")
print("STEP 2: UNCERTAINTY ANALYSIS")
print('='*60)

try:
    uncertainty_analyzer = UncertaintyAnalysis(config)

    uncertainty_params = {
        'wind_resource_variability': {'distribution': 'normal', 'mean': 1.0, 'std': 0.18},
        'arctic_cost_premium': {'distribution': 'triangular', 'low': 1.15, 'mode': 1.25, 'high': 1.4},
        'electricity_price': {'distribution': 'normal', 'mean': 120, 'std': 15},
        'ice_impact_factor': {'distribution': 'beta', 'alpha': 2, 'beta': 8, 'low': 0.0, 'high': 0.15},
        'logistics_cost_multiplier': {'distribution': 'lognormal', 'mean': 1.2, 'sigma': 0.25}
    }

    print("Running Monte Carlo uncertainty analysis...")
    # Pass optimal_design (design variables) not base_design (performance summary)
    uncertainty_results = uncertainty_analyzer.run_monte_carlo(
        baseline_design=optimal_design,
        uncertainty_params=uncertainty_params,
        n_samples=100,  # increase to 1000 for final runs
        reoptimize=False
    )

    uncertainty_summary = uncertainty_results

    print("Uncertainty analysis completed")
    print(f"   Mean LCOE: {uncertainty_summary['mean_lcoe']:.1f}/MWh")
    print(f"   LCOE 95% VaR: {uncertainty_summary['lcoe_var_95']:.1f}/MWh")
    print(f"   Mean NPV: {uncertainty_summary['mean_npv']/1e6:.1f}M")
    print(f"   Probability of Loss: {uncertainty_summary['prob_negative_npv']:.1%}")

except Exception as e:
    print(f"Uncertainty analysis failed: {e}")
    import traceback; traceback.print_exc()

    uncertainty_summary = {
        'mean_lcoe': 102.5, 'lcoe_var_95': 135.0,
        'mean_npv': 18.5e6, 'prob_negative_npv': 0.28, 'npv_var_5': -12e6
    }
    uncertainty_results = uncertainty_summary
    print("Using fallback values for testing")

In [ ]:
print(f"\n{'='*60}")
print("STEP 3: FLEXIBLE DESIGN ANALYSIS")
print('='*60)

try:
    flex_analyzer = FlexibleDesign(config)

    print("Running flexible design analysis...")
    # Pass optimal_design (design variables) using the correct parameter name
    flexibility_results = flex_analyzer.analyze_flexibility(
        baseline_design=optimal_design
    )

    flexibility_summary = flexibility_results

    print("Flexibility analysis completed")
    print(f"   Flexibility Premium: {flexibility_results['flexibility_premium']/1e6:.1f}M")
    print(f"   Most Valuable Option: {flexibility_results['most_valuable_option']}")
    print(f"   Average Options Exercised: {flexibility_results['avg_options_exercised']:.1f}")

except Exception as e:
    print(f"Flexibility analysis failed: {e}")
    import traceback; traceback.print_exc()

    flexibility_summary = {
        'flexibility_premium': 8.5e6, 'most_valuable_option': 'expansion',
        'avg_options_exercised': 1.8, 'expansion_exercise_prob': 0.35,
        'abandonment_exercise_prob': 0.10, 'shutdown_exercise_prob': 0.05
    }
    print("Using fallback values for testing")

In [ ]:
# ============================================================================
# STEP 4: SENSITIVITY ANALYSIS
# ============================================================================

print(f"\n{'='*60}")
print("📊 STEP 4: SENSITIVITY ANALYSIS")
print('='*60)

try:
    # Initialize sensitivity analyzer
    sensitivity_analyzer = SensitivityAnalysis(config)
    
    # Define sensitivity parameters (these will override the defaults in the class)
    sensitivity_params = {
        'wind_capacity_factor': {'base': 0.42, 'range': (0.30, 0.55), 'unit': 'fraction'},
        'arctic_cost_premium': {'base': 1.25, 'range': (1.10, 1.50), 'unit': 'multiplier'},
        'electricity_price': {'base': 120, 'range': (80, 180), 'unit': '£/MWh'},
        'ice_impact_factor': {'base': 0.05, 'range': (0.0, 0.15), 'unit': 'fraction'},
        'discount_rate': {'base': 0.08, 'range': (0.05, 0.12), 'unit': 'fraction'},
        'project_lifetime': {'base': 25, 'range': (20, 30), 'unit': 'years'},
        'logistics_cost_premium': {'base': 1.20, 'range': (1.05, 1.50), 'unit': 'multiplier'}
    }
    
    # Update the sensitivity analyzer's parameters
    sensitivity_analyzer.sensitive_parameters.update({
        k: {'base': v['base'], 'range': v['range']} 
        for k, v in sensitivity_params.items()
    })
    
    print("Running comprehensive sensitivity analysis...")
    
    # Use the correct method name and parameters
    sensitivity_results = sensitivity_analyzer.analyze_sensitivity(
        baseline_design=base_design,
        methods=['local', 'global', 'scenarios'],  # specify methods
        n_samples=1000  # for global sensitivity
    )
    
    print("✅ Sensitivity analysis completed")
    
    # Get parameter rankings from the results
    ranked_sensitivities = sensitivity_results.parameter_rankings
    
    print("   Top 5 Most Sensitive Parameters:")
    for i, (param, sensitivity) in enumerate(ranked_sensitivities[:5], 1):
        print(f"   {i}. {param.replace('_', ' ').title()}: {sensitivity:.2f}")
    
    # Also show local sensitivities if available
    if sensitivity_results.local_sensitivity:
        print("\n   Local Sensitivity Analysis:")
        local_sorted = sorted(sensitivity_results.local_sensitivity.items(), 
                            key=lambda x: abs(x[1]), reverse=True)
        for param, sensitivity in local_sorted[:5]:
            print(f"   - {param.replace('_', ' ').title()}: {sensitivity:.3f}")
    
except Exception as e:
    print(f"❌ Sensitivity analysis failed: {e}")
    print(f"Error details: {type(e).__name__}: {str(e)}")
    
    # Create mock results
    ranked_sensitivities = [
        ('electricity_price', 2.45),
        ('wind_capacity_factor', -1.87),
        ('arctic_cost_premium', -1.23),
        ('ice_impact_factor', -0.98),
        ('discount_rate', -0.76)
    ]
    print("📝 Using mock sensitivity results for testing")

In [ ]:
%matplotlib inline
plt.rcParams['figure.max_open_warning'] = 50

graphics = GraphicsEngine(config)

fig = graphics.create_comprehensive_visualization(
    base_design=base_design,
    uncertainty_summary=uncertainty_summary,  # use uncertainty_summary, always defined
    flexibility_summary=flexibility_summary,
    ranked_sensitivities=ranked_sensitivities
)

plt.show()

In [ ]:
print(f"\n{'='*80}")
print("ALASKA PROJECT - EXECUTIVE SUMMARY")
print('='*80)

print(f"\nFINANCIAL PERFORMANCE:")
print(f"   Base Case LCOE: {base_design['lcoe']:.0f}/MWh")
print(f"   Expected LCOE (with uncertainty): {uncertainty_summary['mean_lcoe']:.0f}/MWh")
print(f"   Base NPV: {base_design['npv']/1e6:.1f}M")
print(f"   Expected NPV: {uncertainty_summary['mean_npv']/1e6:.1f}M")
print(f"   NPV with Flexibility: {(uncertainty_summary['mean_npv'] + flexibility_summary['flexibility_premium'])/1e6:.1f}M")

print(f"\nTECHNICAL PERFORMANCE:")
print(f"   Total Capacity: {base_design['total_capacity']:.2f} MW")
print(f"   Annual Energy: {base_design['annual_energy']:.0f} MWh")
print(f"   Capacity Factor: {base_design['capacity_factor']:.1%}")
print(f"   Wind: {base_design.get('wind_capacity', 0):.2f} MW  Solar: {base_design.get('solar_capacity', 0):.2f} MW  Hydro: {base_design.get('hydro_capacity', 0):.2f} MW")

print(f"\nRISK ASSESSMENT:")
prob_loss = uncertainty_summary['prob_negative_npv']
risk_level = "LOW" if prob_loss < 0.2 else "MODERATE" if prob_loss < 0.4 else "HIGH"
print(f"   Overall Risk Level: {risk_level}")
print(f"   Probability of Loss: {prob_loss:.1%}")
print(f"   LCOE 95% VaR: {uncertainty_summary['lcoe_var_95']:.0f}/MWh")

print(f"\nFLEXIBILITY VALUE:")
print(f"   Flexibility Premium: {flexibility_summary['flexibility_premium']/1e6:.1f}M")
print(f"   Most Valuable Option: {flexibility_summary['most_valuable_option']}")
print(f"   Average Options Exercised: {flexibility_summary['avg_options_exercised']:.1f}")

print(f"\nKEY SENSITIVITIES:")
for i, (param, sensitivity) in enumerate(ranked_sensitivities[:5], 1):
    print(f"   {i}. {param.replace('_', ' ').title()}: {sensitivity:.2f}")

npv_with_flex = uncertainty_summary['mean_npv'] + flexibility_summary['flexibility_premium']
if npv_with_flex > 0 and prob_loss < 0.3:
    recommendation = "PROCEED WITH INVESTMENT"
elif npv_with_flex > 0:
    recommendation = "PROCEED WITH CAUTION"
else:
    recommendation = "DO NOT PROCEED"

print(f"\nRECOMMENDATION: {recommendation}")

print(f"\nARCTIC-SPECIFIC CONSIDERATIONS:")
print(f"   - Sea ice season affects operations (2-4 months)")
print(f"   - Arctic cost premium: +25% above temperate conditions")
print(f"   - Remote logistics require specialised supply chain")
print(f"   - Indigenous community engagement critical")

# Save results using a path relative to the notebook
results_dir = Path("../data/alaska/results")
results_dir.mkdir(parents=True, exist_ok=True)

results_summary = {
    'analysis_date': datetime.now().isoformat(),
    'site': 'Alaska - Igiugig Remote Community',
    'financial_metrics': {
        'base_lcoe': base_design['lcoe'],
        'expected_lcoe': uncertainty_summary['mean_lcoe'],
        'base_npv': base_design['npv'],
        'expected_npv': uncertainty_summary['mean_npv'],
        'npv_with_flexibility': npv_with_flex
    },
    'technical_metrics': {
        'total_capacity_mw': base_design['total_capacity'],
        'annual_energy_mwh': base_design['annual_energy'],
        'capacity_factor': base_design['capacity_factor']
    },
    'risk_metrics': {
        'risk_level': risk_level,
        'prob_negative_npv': prob_loss,
        'lcoe_var_95': uncertainty_summary['lcoe_var_95']
    },
    'flexibility_metrics': {
        'flexibility_premium': flexibility_summary['flexibility_premium'],
        'most_valuable_option': flexibility_summary['most_valuable_option']
    },
    'recommendation': recommendation,
    'top_sensitivities': dict(ranked_sensitivities[:5])
}

output_file = results_dir / f"analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)
print(f"\nResults saved to: {output_file}")

print(f"\n{'='*80}")
print("ALASKA FLEXIMORP ANALYSIS COMPLETE")
print('='*80)

In [ ]:
def run_scenario_comparison():
    """Compare key scenarios using current results."""
    print("\nScenario Comparison:")

    scenarios = {
        'Conservative': {'npv_mult': 0.7, 'risk_mult': 0.5},
        'Base Case':    {'npv_mult': 1.0, 'risk_mult': 1.0},
        'Optimistic':   {'npv_mult': 1.4, 'risk_mult': 1.8},
        'With Flexibility': {'npv_mult': 1.3, 'risk_mult': 0.8}
    }

    base_npv  = uncertainty_summary['mean_npv']
    base_risk = uncertainty_summary['prob_negative_npv']

    for label, m in scenarios.items():
        print(f"   {label:18}: NPV = {base_npv * m['npv_mult']/1e6:6.1f}M,  Risk = {base_risk * m['risk_mult']:5.1%}")

run_scenario_comparison()